# 8.1. Deep Convolutional Neural Networks (AlexNet)
D2L의 Deep Convolutional Neural Networks (AlexNet)장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. AlexNet이 등장하기 전

LeNet을 통해 CNN은 이미 존재했다. 하지만 당시에는 CNN이 컴퓨터 비전의 표준은 아니었다. 전통적인 이미지 처리에선 사람이 직접 특징을 설계했다.

```text
이미지
 ↓
사람이 특징 추출 방법 설계
 ↓
SIFT / SURF / HOG 등으로 특징 추출
 ↓
분류기
 ↓
예측

CNN
이미지
 ↓
CNN
 ↓
특징도 학습
 ↓
분류도 학습
 ↓
예측
```

## 2. Representation Learning

AlexNet이 중요했던 가장 큰 이유는 Representation Learning인 표현 학습을 강력하게 보여줬다는 점이다. 이미지의 낮은 층에서는 비교적 단순한 특징을 학습한다.

```text
낮은 층
→ 선, 경계, 색, 간단한 질감

중간 층
→ 모서리, 패턴, 부분적인 형태

높은 층
→ 눈, 얼굴, 바퀴처럼 더 복잡한 특징

여러 층을 거치면서 단순한 특징들을 조합해 더 복잡한 특징을 만든다.
pixel
 ↓
edge
 ↓
texture / pattern
 ↓
object part
 ↓
object
```

## 3. AlexNet 이전에는 깊은 CNN이 어려웠을까?

크게 세 가지 문제가 있었다.

### 1. 데이터 부족

CNN처럼 파라미터가 많은 모델을 학습시키려면 많은 이미지가 필요하다.

ImageNet과 같은 대규모 이미지 데이터셋이 등장하면서 상황이 달라졌다.

### 2. 계산 성능 부족

Convolution과 행렬곱은 계산량이 많다.

GPU를 이용하면 이런 연산을 대규모로 병렬 처리할 수 있다.

AlexNet은 GPU를 적극적으로 활용했다.

### 3. 학습 기술 부족

이후 다음과 같은 방법들이 등장하면서 깊은 신경망을 안정적으로 학습시키기 쉬워졌다.

```text
좋은 weight initialization
ReLU
Dropout
개선된 optimizer
Data augmentation
```

AlexNet은 이런 변화가 실제 대규모 이미지 분류에서 효과가 있다는 것을 보여준 대표적인 모델이다.

## 4. LeNet과 AlexNet 비교

AlexNet은 LeNet을 더 크고 깊게 만든 모델이라고 보는게 좋다.

| 특징            | LeNet      | AlexNet     |
| ------------- | ---------- | ----------- |
| Convolution 층 | 2개         | 5개          |
| FC 층          | 비교적 작음     | 매우 큼        |
| 활성화 함수        | Sigmoid 계열 | ReLU        |
| Pooling       | 사용         | Max Pooling |
| Dropout       | 없음         | 사용          |
| 입력 이미지        | 작은 이미지     | 큰 이미지       |
| Channel 수     | 적음         | 많음          |

AlexNet은 8개의 학습 가능한 층으로 구성된다. 5개의 Convolution layer + 3개의 Fully Connected layer

## 5. AlexNet 전체 구조

```text
Input
224 × 224
 ↓

Conv 11×11
96 channels
stride=4
 ↓
ReLU
 ↓
MaxPool 3×3
 ↓

Conv 5×5
256 channels
 ↓
ReLU
 ↓
MaxPool 3×3
 ↓

Conv 3×3
384 channels
 ↓
ReLU
 ↓

Conv 3×3
384 channels
 ↓
ReLU
 ↓

Conv 3×3
256 channels
 ↓
ReLU
 ↓
MaxPool 3×3
 ↓

Flatten
 ↓
Linear 4096
 ↓
ReLU
 ↓
Dropout
 ↓

Linear 4096
 ↓
ReLU
 ↓
Dropout
 ↓

Linear
 ↓
class prediction
```

## 6. 첫 번째 Conv가 11 x 11인 이유

AlexNet의 첫 번째 convolution은 상당히 크다.

```py
nn.Conv2d(
    in_channels=1,
    out_channels=96,
    kernel_size=11,
    stride=4,
    padding=1
)
```

ImageNet 이미지가 MNIST보다 훨씬 크고 복잡했기 때문에 초반부터 넓은 영역을 보도록 큰 커널을 사용했다. 그리고 stride=4이기 때문에 이미지의 H, W가 빠르게 감소한다.

입력이 이럴때 [1, 1, 224, 224] -> [1, 96, 54, 54] 가 된다.

## 7. AlexNet의 채널 변화

AlexNet에서 Convolution 부분만 보면 이렇다.

```text
입력
1 × 224 × 224

↓ Conv

96 × 54 × 54

↓ Pooling

96 × 26 × 26

↓ Conv

256 × 26 × 26

↓ Pooling

256 × 12 × 12

↓ Conv

384 × 12 × 12

↓ Conv

384 × 12 × 12

↓ Conv

256 × 12 × 12

↓ Pooling

256 × 5 × 5
```

이미지가 작아진다고 해서 정보가 단순히 없어지는 것은 아니다. 뒤로 갈수록 feature map은 작아지고 channel은 많아진다. 각 channel은 학습된 서로 다른 특징을 나타낸다. 

마지막에는 256 x 5 x 5가 되는데 펼쳐서 6400개의 feature로 만든다. [batch, 6400]

## 8. Sigmoid 대신 ReLU

AlexNet의 중요한 변화 중 하나는 활성화 함수로 ReLU를 사용한 것이다.

Sigmoid는 값이 너무 크거나 작아지면 gradient가 거의 0에 가까워질 수 있다.

```text
Sigmoid

출력 ≈ 0 또는 1
        ↓
gradient ≈ 0
        ↓
앞쪽 layer까지 gradient 전달 어려움


ReLU

x > 0
↓
gradient = 1
```

그래서 당시에 깊은 네트워크를 학습시키는 데 큰 도움이 됬다고 한다.

## 9. Dropout으로 Overfitting 방지

AlexNet에는 매우 큰 Fully Connected Layer가 존재한다.

    6400 -> 4096 -> 4096 -> output

파라미터 수가 엄청나게 많아진다. 그래서 overfitting 위험도 커진다. 그래서 이를 줄이기 위해 Dropout을 사용했다.

AlexNet은 또한 이미지 뒤집기, 잘라내기, 색 변화 등의 Data Augmentation도 적극적으로 사용했다.

## 10. AlexNet PyTorch 구현

D2L 클래스를 사용하지 않고 PyTorch 방식으로 작성하면 이렇다.

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.net = nn.Sequential(

            # 1번째 Conv
            nn.Conv2d(
                in_channels=1, # FashionMNIST는 흑백 이미지이므로 in_channels=1
                out_channels=96,
                kernel_size=11,
                stride=4,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # 2번째 Conv
            nn.Conv2d(
                96,
                256,
                kernel_size=5,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # 3번째 Conv
            nn.Conv2d(
                256,
                384,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            # 4번째 Conv
            nn.Conv2d(
                384,
                384,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            # 5번째 Conv
            nn.Conv2d(
                384,
                256,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Fully Connected Layer로 전달
            nn.Flatten(),

            nn.Linear(256 * 5 * 5, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        return self.net(x)

## 11. 각 Layer의 출력 크기 확인하기

In [4]:
model = AlexNet()

X = torch.randn(1, 1, 224, 224)

for layer in model.net:
    X = layer(X)

    print(
        layer.__class__.__name__,
        X.shape
    )

Conv2d torch.Size([1, 96, 54, 54])
ReLU torch.Size([1, 96, 54, 54])
MaxPool2d torch.Size([1, 96, 26, 26])
Conv2d torch.Size([1, 256, 26, 26])
ReLU torch.Size([1, 256, 26, 26])
MaxPool2d torch.Size([1, 256, 12, 12])
Conv2d torch.Size([1, 384, 12, 12])
ReLU torch.Size([1, 384, 12, 12])
Conv2d torch.Size([1, 384, 12, 12])
ReLU torch.Size([1, 384, 12, 12])
Conv2d torch.Size([1, 256, 12, 12])
ReLU torch.Size([1, 256, 12, 12])
MaxPool2d torch.Size([1, 256, 5, 5])
Flatten torch.Size([1, 6400])
Linear torch.Size([1, 4096])
ReLU torch.Size([1, 4096])
Dropout torch.Size([1, 4096])
Linear torch.Size([1, 4096])
ReLU torch.Size([1, 4096])
Dropout torch.Size([1, 4096])
Linear torch.Size([1, 10])


결국 CNN은 이미지 -> 특징 벡터 -> 클래스 점수 로 변환하는 과정이다.

## 12. Fashion-MNIST에선 왜 224 x 224로 키우나?

Fashion-MNIST 이미지는 원래 28 x 28이다. 하지만 AlexNet은 큰 이미지를 처리하도록 만들어졌다. 커진다고 이미지에 새로운 정보가 생기는 건 아니다. 계산량이 증가한다. AlexNet 구조를 실습하기 위해 사용하는 것이다.

In [5]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

100.0%
100.0%
100.0%
100.0%


## 13. AlexNet의 단점

AlexNet은 역사적으로 중요한 모델이지만 지금 기준으로는 비효율적이라고 한다.

특히 Fully Connected Layer가 너무 크다. 6400 -> 4096

첫 번째 FC만 해도 파라미터가 대략 6400 x 4096 = 26214400

두 번째까지 합치면 FC 부분에서 수천만 개의 파라미터가 발생한다. 그래서 이후 등장하는 CNN들은 이런 비효율을 크게 개선한다.

## 14. 오늘의 정리

- AlexNet은 2012년 ImageNet에서 딥러닝 CNN의 가능성을 강하게 보여준 모델이다.
- AlexNet은 완전히 새로운 구조라기보다 LeNet을 훨씬 깊고 크게 확장한 구조다.
- AlexNet은 5 Conv + 3 FC의 총 8개 학습 layer를 가진다.
- CNN이 직접 이미지의 특징을 학습하는 Representation Learning이 핵심이다.
- 앞부분에서는 H, W가 감소하고 channel 수가 증가한다.
- AlexNet은 Sigmoid 대신 ReLU를 사용하여 깊은 네트워크를 더 쉽게 학습했다.
- 큰 Fully Connected Layer의 overfitting을 줄이기 위해 Dropout을 사용했다.
- Data Augmentation도 적극적으로 사용했다.
- Fashion-MNIST의 28×28 이미지를 224×224로 확대하는 것은 AlexNet 구조 실습을 위한 것이며 새로운 정보가 생기는 것은 아니다.
- AlexNet은 FC layer의 파라미터가 지나치게 많아 현대 기준으로는 비효율적이다.
- AlexNet에서 가장 중요하게 볼 것은 이미지 → 여러 단계의 특징 추출 → 특징 벡터 → 분류라는 전체 흐름이다.